# Week 7: SOLID Principles (Practical) -- SRP & OCP — PHASE 3: Making Composition Maintainable

*Object Oriented Programming . 3 Hours . Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Explain what SOLID design principles are and why they matter
2. Apply the **Single Responsibility Principle (SRP)** to keep classes focused
3. Recognize common **code smells** that signal SRP violations
4. Apply the **Open/Closed Principle (OCP)** to write extensible code
5. Refactor a "God class" into well-structured, single-purpose classes
6. Extend system behavior **without modifying** existing code

## 🎯 Core Mastery Connection

SRP means each component does ONE thing. OCP means you extend by adding new components, not modifying existing ones. Together, these principles ensure that your composed system stays maintainable as it grows — each piece remains small, focused, and safe to change independently.

---

## Part 1: What Are Design Principles?

Design principles are **guidelines** that help us write code that is:

- Easy to **understand**
- Easy to **change**
- Easy to **test**

The **SOLID** principles are five well-known design principles for object-oriented programming.

| Letter | Principle | One-Line Summary | Coverage |
|--------|-----------|------------------|----------|
| **S** | Single Responsibility | A class should have only one reason to change | This week |
| **O** | Open/Closed | Open for extension, closed for modification | This week |
| **L** | Liskov Substitution | Subtypes must be usable in place of their parent type | Advanced — self-study |
| **I** | Interface Segregation | Don't force classes to depend on methods they don't use | Advanced — self-study |
| **D** | Dependency Inversion | Depend on abstractions, not concrete details | Advanced — self-study |

In this course, we focus on **S** and **O** as they are the most practical for the projects you will build. **L**, **I**, and **D** are important in larger systems — you can explore them as you advance in your career.

> **Think of it like engineering:** A well-designed circuit board has separate modules for power, logic, and communication. You wouldn't put everything on one chip if you want to maintain or upgrade the system later.

---

## Part 2: Single Responsibility Principle (SRP)

### The Rule

> **A class should have only one reason to change.**

This means each class should do **one job** and do it well.

### A Bad Example

Imagine a class that handles a temperature sensor in a mechatronics system. It reads the sensor, stores the data, AND sends an email alert. That is three jobs in one class!

**Figure 2.1** -- A class doing too many things

```
+---------------------------+
|   TemperatureSensor       |
+---------------------------+
| - temperature             |
| - log_file                |
| - email_address           |
+---------------------------+
| + read_temperature()      |  <-- Sensor job
| + save_to_file()          |  <-- Storage job
| + send_alert_email()      |  <-- Notification job
+---------------------------+
```

In [ ]:
# BAD DESIGN: This class does too many things!

class TemperatureSensor:
    """A class that reads temperature, saves data, and sends alerts.
    This violates SRP because it has THREE reasons to change."""

    def __init__(self, sensor_id, log_file, email):
        self.sensor_id = sensor_id
        self.log_file = log_file
        self.email = email
        self.temperature = 0.0

    def read_temperature(self):
        # Simulating a sensor reading
        import random
        self.temperature = round(random.uniform(15.0, 85.0), 1)
        return self.temperature

    def save_to_file(self):
        # Storage logic
        print(f"Saving {self.temperature}C to {self.log_file}")

    def send_alert_email(self):
        # Notification logic
        if self.temperature > 70.0:
            print(f"ALERT email to {self.email}: Temp is {self.temperature}C!")


# Using the bad design
sensor = TemperatureSensor("S-001", "log.txt", "engineer@factory.com")
sensor.read_temperature()
sensor.save_to_file()
sensor.send_alert_email()

### Why Is This Bad?

If you change **how data is stored** (e.g., switch from file to database), you have to modify the `TemperatureSensor` class. But that class also handles reading and alerting -- you might accidentally break something unrelated.

| Reason to Change | Responsibility |
|-------------------|---------------|
| Sensor hardware changes | Reading temperature |
| Storage format changes | Saving data |
| Alert method changes | Sending notifications |

Three reasons to change = three responsibilities = **SRP violation**.

### The Fix: One Class, One Job

**Figure 2.2** -- Classes with single responsibilities

```
+--------------------+    +--------------------+    +--------------------+
| TemperatureSensor  |    | DataLogger         |    | AlertNotifier      |
+--------------------+    +--------------------+    +--------------------+
| + read()           |    | + save(data)       |    | + check_and_alert()|
+--------------------+    +--------------------+    +--------------------+
```

In [ ]:
# GOOD DESIGN: Each class has one job

import random

class TemperatureSensor:
    """Only reads temperature. That's it."""

    def __init__(self, sensor_id):
        self.sensor_id = sensor_id

    def read(self):
        return round(random.uniform(15.0, 85.0), 1)


class DataLogger:
    """Only saves data to a file."""

    def __init__(self, log_file):
        self.log_file = log_file

    def save(self, sensor_id, value):
        print(f"Saving {sensor_id}: {value}C to {self.log_file}")


class AlertNotifier:
    """Only sends alerts when needed."""

    def __init__(self, email, threshold=70.0):
        self.email = email
        self.threshold = threshold

    def check_and_alert(self, value):
        if value > self.threshold:
            print(f"ALERT to {self.email}: Temperature {value}C exceeds {self.threshold}C!")


# Using the good design
sensor = TemperatureSensor("S-001")
logger = DataLogger("log.txt")
notifier = AlertNotifier("engineer@factory.com")

temp = sensor.read()
print(f"Temperature reading: {temp}C")
logger.save(sensor.sensor_id, temp)
notifier.check_and_alert(temp)

---

## Part 3: Spotting SRP Violations (Code Smells)

A **code smell** is a sign that something might be wrong with your design. Here are common smells that indicate SRP violations:

| Code Smell | What It Looks Like | Why It's Bad |
|------------|--------------------|--------------|
| **Long class** | Class has 200+ lines | Probably doing too many things |
| **Many methods** | Class has 10+ methods doing different tasks | Multiple responsibilities |
| **"And" in description** | "This class reads sensors AND logs data AND sends alerts" | Each "and" = another responsibility |
| **Too many imports** | Class imports email, file, database, GUI libraries | Different domains mixed together |
| **Hard to name** | You call it `Manager`, `Handler`, `Processor`, `Utility` | Vague name = vague purpose |

### Quick Test

Try to describe your class in **one sentence without using "and"**. If you can't, it probably violates SRP.

- **Bad:** "This class reads the motor speed **and** logs it **and** displays it on screen."
- **Good:** "This class reads the motor speed."

In [ ]:
# EXERCISE: Can you spot the SRP violations?

class RobotController:
    """Controls a robot arm, saves movement logs, and draws the arm on screen."""

    def __init__(self):
        self.position = [0, 0, 0]  # x, y, z
        self.log = []

    def move_to(self, x, y, z):
        self.position = [x, y, z]
        self.log.append(f"Moved to {x}, {y}, {z}")
        print(f"Robot moved to ({x}, {y}, {z})")

    def save_log(self, filename):
        print(f"Saving {len(self.log)} entries to {filename}")

    def draw_robot(self):
        print(f"Drawing robot at position {self.position}")

    def calculate_distance(self, target):
        # Math calculation
        return sum((a - b) ** 2 for a, b in zip(self.position, target)) ** 0.5


# This class has at least 3 responsibilities:
# 1. Movement control
# 2. Logging
# 3. Visualization
robot = RobotController()
robot.move_to(10, 20, 5)
robot.draw_robot()
robot.save_log("robot_log.txt")

---

## Part 4: Refactoring -- One Class, One Job

**Refactoring** means improving your code's structure without changing what it does.

Let's refactor the `RobotController` from Part 3.

**Figure 4.1** -- Refactored robot system

```
+------------------+    +------------------+    +------------------+
| RobotArm         |    | MovementLogger   |    | RobotRenderer    |
+------------------+    +------------------+    +------------------+
| - position       |    | - entries        |    | + draw(position) |
| + move_to(x,y,z) |    | + log(message)   |    +------------------+
| + distance(tgt)  |    | + save(filename) |
+------------------+    +------------------+
```

In [ ]:
# GOOD DESIGN: Refactored robot system

class RobotArm:
    """Controls the robot arm movement."""

    def __init__(self):
        self.position = [0, 0, 0]

    def move_to(self, x, y, z):
        self.position = [x, y, z]
        print(f"Robot moved to ({x}, {y}, {z})")

    def distance_to(self, target):
        return sum((a - b) ** 2 for a, b in zip(self.position, target)) ** 0.5


class MovementLogger:
    """Logs robot movements."""

    def __init__(self):
        self.entries = []

    def log(self, message):
        self.entries.append(message)
        print(f"Logged: {message}")

    def save(self, filename):
        print(f"Saving {len(self.entries)} entries to {filename}")


class RobotRenderer:
    """Draws the robot on screen."""

    def draw(self, position):
        print(f"Drawing robot at position {position}")


# Using the refactored design
arm = RobotArm()
logger = MovementLogger()
renderer = RobotRenderer()

arm.move_to(10, 20, 5)
logger.log(f"Moved to {arm.position}")
renderer.draw(arm.position)
logger.save("robot_log.txt")

---

## Part 5: Open/Closed Principle (OCP)

### The Rule

> **Software entities should be open for extension, but closed for modification.**

This means:
- **Open for extension:** You can add new behavior
- **Closed for modification:** You don't have to change existing code to add it

### A Bad Example

Imagine a system that calculates the area of different shapes for a CNC cutting machine.

In [ ]:
# BAD DESIGN: Violates OCP
# Every time we add a new shape, we must MODIFY this function

class AreaCalculator:
    """Calculates cutting area for CNC machine."""

    def calculate(self, shape_type, **kwargs):
        if shape_type == "rectangle":
            return kwargs["width"] * kwargs["height"]
        elif shape_type == "circle":
            import math
            return math.pi * kwargs["radius"] ** 2
        elif shape_type == "triangle":
            return 0.5 * kwargs["base"] * kwargs["height"]
        # To add a new shape, we MUST modify this method!
        # What if we need hexagon? Pentagon? Custom shape?
        else:
            raise ValueError(f"Unknown shape: {shape_type}")


calc = AreaCalculator()
print(f"Rectangle: {calc.calculate('rectangle', width=10, height=5)} sq cm")
print(f"Circle: {calc.calculate('circle', radius=7):.2f} sq cm")

### Why Is This Bad?

Every time you add a new shape, you **modify** the `calculate` method. This can:
- Introduce bugs in existing shape calculations
- Make the method longer and harder to read
- Require testing ALL shapes again after each change

| Problem | Impact |
|---------|--------|
| Growing if/elif chain | Hard to read and maintain |
| Must modify existing code | Risk of breaking working features |
| All shapes in one place | Cannot distribute work among team members |

---

## Part 6: Extending Without Modifying

The solution is to use **inheritance** (or composition). Each shape is its own class with an `area()` method. To add a new shape, you just create a new class -- no existing code is changed.

**Figure 6.1** -- OCP-compliant shape system

```
        +------------------+
        |   Shape (base)   |
        +------------------+
        | + area()         |
        +------------------+
               /    |    \
              /     |     \
+------------+ +--------+ +------------+
| Rectangle  | | Circle | | Triangle   |
+------------+ +--------+ +------------+
| + area()   | |+ area()| | + area()   |
+------------+ +--------+ +------------+
```

Adding a `Hexagon`? Just create a new class. **No existing code changes.**

In [ ]:
# GOOD DESIGN: Follows OCP
# To add a new shape, just create a new class

import math

class Shape:
    """Base class for all cutting shapes."""

    def area(self):
        raise NotImplementedError("Subclasses must implement area()")


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return 0.5 * self.base * self.height


# The calculator works with ANY shape -- no modification needed
def total_cutting_area(shapes):
    """Calculate total area for all shapes. Works with any Shape subclass."""
    return sum(s.area() for s in shapes)


# Using the good design
shapes = [
    Rectangle(10, 5),
    Circle(7),
    Triangle(8, 6),
]

for shape in shapes:
    print(f"{shape.__class__.__name__}: {shape.area():.2f} sq cm")

print(f"Total cutting area: {total_cutting_area(shapes):.2f} sq cm")

In [ ]:
# EXTENDING: Adding a new shape WITHOUT modifying any existing code

class Hexagon(Shape):
    """Regular hexagon -- added without changing any existing class!"""

    def __init__(self, side_length):
        self.side_length = side_length

    def area(self):
        return (3 * math.sqrt(3) / 2) * self.side_length ** 2


# It works perfectly with the existing function
shapes.append(Hexagon(4))

for shape in shapes:
    print(f"{shape.__class__.__name__}: {shape.area():.2f} sq cm")

print(f"Total cutting area: {total_cutting_area(shapes):.2f} sq cm")

### SRP + OCP Together

Notice how these two principles work together:

| Principle | What It Tells Us |
|-----------|------------------|
| **SRP** | Each class does one thing |
| **OCP** | Add new things by adding new classes, not by editing old ones |

When your classes follow SRP, it becomes much easier to follow OCP too!

---

## Part 7: Practical Refactoring Exercise

Let's look at a real-world-ish example: a motor control system that has grown into a "God class".

**Figure 7.1** -- A God class that does everything

In [ ]:
# A "God Class" -- does EVERYTHING related to the motor

class MotorSystem:
    """Controls motor, logs data, checks safety, and generates reports."""

    def __init__(self, motor_id):
        self.motor_id = motor_id
        self.speed = 0
        self.max_speed = 5000  # RPM
        self.log = []

    # --- Motor control ---
    def set_speed(self, rpm):
        self.speed = rpm
        self.log.append(f"Speed set to {rpm}")
        print(f"Motor {self.motor_id}: speed set to {rpm} RPM")

    def stop(self):
        self.speed = 0
        self.log.append("Motor stopped")
        print(f"Motor {self.motor_id}: STOPPED")

    # --- Safety checks ---
    def check_overspeed(self):
        if self.speed > self.max_speed:
            print(f"WARNING: {self.speed} exceeds max {self.max_speed}!")
            return True
        return False

    # --- Logging ---
    def save_log(self, filename):
        print(f"Saving {len(self.log)} log entries to {filename}")

    # --- Reporting ---
    def generate_report(self):
        print(f"=== Motor Report for {self.motor_id} ===")
        print(f"Current speed: {self.speed} RPM")
        print(f"Log entries: {len(self.log)}")


# Using the God class
motor = MotorSystem("M-001")
motor.set_speed(3000)
motor.check_overspeed()
motor.generate_report()
motor.save_log("motor_log.txt")

---

## Exercises

> **Composition lens:** Every time you split a God class into focused components (SRP) or add new behavior without modifying existing code (OCP), you are making composition maintainable. A system of small, single-purpose components is far easier to compose, test, and extend.

Complete the following exercises. Each exercise builds on what you learned today.

---

## 🎢 Exercises

Complete the following exercises. Each exercise builds on what you learned today.

### Exercise 1: Identify SRP Violations (Easy)

The class below violates SRP. List ALL responsibilities this class has (write as comments). How many reasons to change does it have?

<details>
<summary>💡 Hint</summary>
Count how many <em>different jobs</em> the class does. If you can describe it only with 'and', it violates SRP.
</details>

In [ ]:
# ✏️ [EX1]
# The class below violates SRP.
# List ALL responsibilities this class has (write as comments).
# How many reasons to change does it have?

class ConveyorBelt:
    def __init__(self, belt_id):
        self.belt_id = belt_id
        self.speed = 0
        self.items_count = 0
        self.log = []

    def set_speed(self, speed):
        self.speed = speed

    def count_item(self):
        self.items_count += 1

    def save_log(self, filename):
        print(f"Saving to {filename}")

    def display_status(self):
        print(f"Belt {self.belt_id}: speed={self.speed}, items={self.items_count}")

    def send_maintenance_alert(self, email):
        print(f"Sending alert to {email}")

# YOUR ANSWER:
# Responsibility 1: ...
# Responsibility 2: ...
# ...
# Number of reasons to change: ...

### Exercise 2: Refactor the ConveyorBelt (Medium)

Split the ConveyorBelt class from EX1 into separate classes. Each class should have exactly ONE responsibility. Then create instances and show them working together.

<details>
<summary>💡 Hint</summary>
Create a separate class for each responsibility you found in EX1. Each class should have a single, clear purpose.
</details>

In [ ]:
# ✏️ [EX2]
# Split the ConveyorBelt class from EX1 into separate classes.
# Each class should have exactly ONE responsibility.
# Then create instances and show them working together.

# Hint: Think about what the separate jobs are from EX1

# YOUR CODE HERE


### Exercise 3: Describe in One Sentence (Easy)

For each class below, write a ONE-sentence description WITHOUT using "and". If you can't, the class probably violates SRP. Class A: Reads sensor data and writes it to a CSV file Class B: Validates user input from a form Class C: Manages database connections, runs queries, and formats results as HTML

<details>
<summary>💡 Hint</summary>
If your sentence needs the word <em>'and'</em>, the class probably has too many responsibilities.
</details>

In [ ]:
# ✏️ [EX3]
# For each class below, write a ONE-sentence description WITHOUT using "and".
# If you can't, the class probably violates SRP.
#
# Class A: Reads sensor data and writes it to a CSV file
# Class B: Validates user input from a form
# Class C: Manages database connections, runs queries, and formats results as HTML

# YOUR ANSWERS:
# Class A: Violates SRP? ...
# Class B: Violates SRP? ...
# Class C: Violates SRP? ...

### Exercise 4: OCP with Sensors (Medium)

Create a base class `Sensor` with a method `read_value()`. Then create three subclasses: - TemperatureSensor (returns a value between 20 and 100) - PressureSensor (returns a value between 1 and 10) - HumiditySensor (returns a value between 30 and 90) Write a function `read_all_sensors(sensors)` that works with ANY sensor type. Hint: Use random.uniform() for simulated readings

<details>
<summary>💡 Hint</summary>
Create a base <code>Sensor</code> class with a <code>read_value()</code> method. Each subclass overrides it with its own logic.
</details>

In [ ]:
# ✏️ [EX4]
# Create a base class `Sensor` with a method `read_value()`.
# Then create three subclasses:
# - TemperatureSensor (returns a value between 20 and 100)
# - PressureSensor (returns a value between 1 and 10)
# - HumiditySensor (returns a value between 30 and 90)
#
# Write a function `read_all_sensors(sensors)` that works with ANY sensor type.
#
# Hint: Use random.uniform() for simulated readings

import random

# YOUR CODE HERE


### Exercise 5: Extend Without Modifying (Easy)

Using your sensor system from EX4, add a new sensor type: - VibrationSensor (returns a value between 0 and 50) The key: you should NOT modify any existing class or function. Just add the new class and show it works with read_all_sensors().

<details>
<summary>💡 Hint</summary>
Just create a new subclass — you should NOT need to change any existing class code.
</details>

In [ ]:
# ✏️ [EX5]
# Using your sensor system from EX4, add a new sensor type:
# - VibrationSensor (returns a value between 0 and 50)
#
# The key: you should NOT modify any existing class or function.
# Just add the new class and show it works with read_all_sensors().

# YOUR CODE HERE


### Exercise 6: OCP with Notifications (Medium)

Create a notification system that follows OCP. Base class: Notifier with method send(message) Subclasses: - EmailNotifier: prints "Email: <message>" - SMSNotifier: prints "SMS: <message>" - BuzzerNotifier: prints "BUZZ! <message>" Write a function `alert_all(notifiers, message)` that sends the message through all notifiers. Hint: Same pattern as the Shape example

<details>
<summary>💡 Hint</summary>
Base class <code>Notifier</code> with <code>send(message)</code>. Subclasses: <code>EmailNotifier</code>, <code>SMSNotifier</code>, <code>PushNotifier</code>.
</details>

In [ ]:
# ✏️ [EX6]
# Create a notification system that follows OCP.
#
# Base class: Notifier with method send(message)
# Subclasses:
#   - EmailNotifier: prints "Email: <message>"
#   - SMSNotifier: prints "SMS: <message>"
#   - BuzzerNotifier: prints "BUZZ! <message>"
#
# Write a function `alert_all(notifiers, message)` that sends
# the message through all notifiers.
#
# Hint: Same pattern as the Shape example

# YOUR CODE HERE


### Exercise 7: Spot the OCP Violation (Medium)

The function below violates OCP. Explain WHY it violates OCP, then rewrite it using classes so it follows OCP.

<details>
<summary>💡 Hint</summary>
Look for <code>if/elif</code> chains that check types. These break OCP because adding a new type requires modifying the function.
</details>

In [ ]:
# ✏️ [EX7]
# The function below violates OCP. Explain WHY it violates OCP,
# then rewrite it using classes so it follows OCP.

def calculate_power(motor_type, voltage, current):
    if motor_type == "DC":
        return voltage * current
    elif motor_type == "AC_single":
        power_factor = 0.85
        return voltage * current * power_factor
    elif motor_type == "AC_three":
        power_factor = 0.9
        return 1.732 * voltage * current * power_factor
    else:
        return 0

# Why it violates OCP: ...

# YOUR REFACTORED CODE HERE


### Exercise 8: SRP Refactoring Practice (Medium)

The class below is a "God class" for a simple weather station. Refactor it into at least 3 separate classes that each follow SRP.

<details>
<summary>💡 Hint</summary>
Separate data collection, data processing, and data display into different classes.
</details>

In [ ]:
# ✏️ [EX8]
# The class below is a "God class" for a simple weather station.
# Refactor it into at least 3 separate classes that each follow SRP.

class WeatherStation:
    def __init__(self, station_id):
        self.station_id = station_id
        self.temperature = 0
        self.humidity = 0
        self.readings = []

    def read_sensors(self):
        import random
        self.temperature = round(random.uniform(-10, 40), 1)
        self.humidity = round(random.uniform(20, 95), 1)
        self.readings.append((self.temperature, self.humidity))

    def save_to_csv(self, filename):
        print(f"Saving {len(self.readings)} readings to {filename}")

    def display_dashboard(self):
        print(f"Station {self.station_id}")
        print(f"  Temp: {self.temperature}C")
        print(f"  Humidity: {self.humidity}%")

    def check_frost_warning(self):
        if self.temperature < 0:
            print("FROST WARNING!")

# YOUR REFACTORED CODE HERE


### Exercise 9: Design from Scratch (Challenge)

You are building a simple parking garage system. It needs to: 1. Track which parking spots are occupied 2. Calculate parking fees based on duration 3. Print receipts Design this system using SEPARATE classes that follow SRP. Name your classes and list their methods (you don't need full implementation). Then implement at least the basic structure with __init__ and method signatures. Hint: Think about what changes independently

<details>
<summary>💡 Hint</summary>
Think about what objects exist in a parking garage: <code>ParkingSpot</code>, <code>ParkingGarage</code>, <code>Ticket</code>. Each has one job.
</details>

In [ ]:
# ✏️ [EX9]
# You are building a simple parking garage system.
# It needs to:
#   1. Track which parking spots are occupied
#   2. Calculate parking fees based on duration
#   3. Print receipts
#
# Design this system using SEPARATE classes that follow SRP.
# Name your classes and list their methods (you don't need full implementation).
# Then implement at least the basic structure with __init__ and method signatures.
#
# Hint: Think about what changes independently

# YOUR CODE HERE


### Exercise 10: OCP with Data Export (Challenge)

Create a system that can export sensor data in different formats. Base class: DataExporter with method export(data) Subclasses: - CSVExporter: prints data as comma-separated values - JSONExporter: prints data as a JSON-like string - TableExporter: prints data in a simple table format Test with sample data: [("S1", 25.3), ("S2", 30.1), ("S3", 22.8)] The key: you should be able to add a new format (e.g., XMLExporter) without changing any existing code.

<details>
<summary>💡 Hint</summary>
Abstract base: <code>DataExporter</code> with <code>export(data)</code>. Subclasses: <code>CSVExporter</code>, <code>JSONExporter</code>, <code>HTMLExporter</code>.
</details>

In [ ]:
# ✏️ [EX10]
# Create a system that can export sensor data in different formats.
#
# Base class: DataExporter with method export(data)
# Subclasses:
#   - CSVExporter: prints data as comma-separated values
#   - JSONExporter: prints data as a JSON-like string
#   - TableExporter: prints data in a simple table format
#
# Test with sample data: [("S1", 25.3), ("S2", 30.1), ("S3", 22.8)]
#
# The key: you should be able to add a new format (e.g., XMLExporter)
# without changing any existing code.

# YOUR CODE HERE


### Exercise 11: Combined SRP + OCP (Challenge)

A factory has different types of quality checks: - DimensionCheck: checks if a part's size is within tolerance - WeightCheck: checks if a part's weight is within range - ColorCheck: checks if a part's color matches the expected color Design a system where: 1. Each check is its own class (SRP) 2. New check types can be added without modifying existing code (OCP) 3. A function `run_all_checks(part, checks)` runs all checks on a part Hint: Create a base QualityCheck class with a check(part) method

<details>
<summary>💡 Hint</summary>
Each quality check should be a separate class (SRP). New check types should be addable without changing existing code (OCP).
</details>

In [ ]:
# ✏️ [EX11]
# A factory has different types of quality checks:
#   - DimensionCheck: checks if a part's size is within tolerance
#   - WeightCheck: checks if a part's weight is within range
#   - ColorCheck: checks if a part's color matches the expected color
#
# Design a system where:
#   1. Each check is its own class (SRP)
#   2. New check types can be added without modifying existing code (OCP)
#   3. A function `run_all_checks(part, checks)` runs all checks on a part
#
# Hint: Create a base QualityCheck class with a check(part) method

# YOUR CODE HERE


### Exercise 12: Reflection (Easy)

Answer these questions as comments: 1. In your own words, what is the Single Responsibility Principle? 2. Give a real-world (non-code) example of SRP. (e.g., a kitchen tool that does one thing vs. a Swiss Army knife) 3. In your own words, what is the Open/Closed Principle? 4. Why do SRP and OCP work well together?

<details>
<summary>💡 Hint</summary>
Think about real-world examples. A chef (SRP: only cooks) vs. a restaurant (OCP: can add new dishes without rebuilding the kitchen).
</details>

In [ ]:
# ✏️ [EX12]
# Answer these questions as comments:
#
# 1. In your own words, what is the Single Responsibility Principle?
#
# 2. Give a real-world (non-code) example of SRP.
#    (e.g., a kitchen tool that does one thing vs. a Swiss Army knife)
#
# 3. In your own words, what is the Open/Closed Principle?
#
# 4. Why do SRP and OCP work well together?

# YOUR ANSWERS:
# 1. ...
# 2. ...
# 3. ...
# 4. ...

---

### 🌉 Bridge to Next Week

This week we learned how to **structure** our classes well using SRP and OCP. Next week, we will learn about **Custom Exceptions** -- how to create your own error types that make debugging and error handling much clearer in your programs.

Think about this: when a motor overheats, should you raise a generic `Exception` or a specific `MotorOverheatError`? Next week you will find out!

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_07"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")